# Hafta 5 — Doğrusal Regresyon

Elle → NumPy → scikit-learn sırasıyla ilerliyoruz. Sonunda 4. haftanın taban modelini yeneceğiz.

Veri: `elektrik_tuketimi.csv` (ve ödev için `pv_uretim.csv`).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def rapor(ad, y, yp):
    print(f"{ad:34s} MAE={mean_absolute_error(y, yp):6.2f}  RMSE={np.sqrt(mean_squared_error(y, yp)):6.2f}  R2={r2_score(y, yp):6.3f}")

df = pd.read_csv("elektrik_tuketimi.csv")
df = df[df.tuketim_kW < 300].copy(); df["sicaklik_C"] = df["sicaklik_C"].interpolate()
egitim, test = df[df.gun <= 24], df[df.gun > 24]
print(df.shape)

## 1. Basit regresyon: kapalı çözüm (Örnek 5.1)

In [ ]:
x = np.array([20, 24, 28, 32.]); y = np.array([80, 90, 110, 140.])
w = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean())**2).sum()
b = y.mean() - w * x.mean()
yp = w*x + b
print(f"w = {w}, b = {b}, RMSE = {np.sqrt(np.mean((y-yp)**2)):.2f}, R2 = {1 - ((y-yp)**2).sum()/((y-y.mean())**2).sum():.3f}")
print("30 °C ->", w*30 + b, "kW")

plt.scatter(x, y, color="k", zorder=3); xx = np.linspace(18, 34, 50); plt.plot(xx, w*xx + b)
for xi, yi in zip(x, y): plt.plot([xi, xi], [yi, w*xi+b], "r")
plt.xlabel("°C"); plt.ylabel("kW"); plt.grid(alpha=.3); plt.show()

## 2. Gradyan inişi: kendimiz yazalım (Örnek 5.2)

In [ ]:
def gradyan_inisi(x, y, eta=0.1, iter=100, w=0.0, b=0.0, yazdir=(1, 2, 10, 100)):
    gecmis = []
    for it in range(1, iter + 1):
        yp = w*x + b
        gw = 2*np.mean((yp - y) * x); gb = 2*np.mean(yp - y)   # türevler
        w -= eta*gw; b -= eta*gb
        gecmis.append(np.mean((w*x + b - y)**2))
        if it in yazdir: print(f"it={it:3d}  w={w:.4f}  b={b:.4f}  MSE={gecmis[-1]:.4f}")
    return w, b, gecmis

w_, b_, g = gradyan_inisi(np.array([-1., 1.]), np.array([1., 3.]))
print("kapalı çözüm: w=1, b=2")

**Deneyin:** `eta` = 0.02, 0.5 ve 1.1 için `gradyan_inisi`'ni çağırıp MSE geçmişini aynı grafikte (log ekseni) çizin.

In [ ]:
plt.figure(figsize=(7, 3.5))
for eta in (0.02, 0.1, 0.5, 1.05):
    _, _, g = gradyan_inisi(np.array([-1., 1.]), np.array([1., 3.]), eta=eta, iter=40, yazdir=())
    plt.plot(g, label=f"η={eta}")
plt.yscale("log"); plt.xlabel("iterasyon"); plt.ylabel("MSE"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 3. Ölçeklemenin gradyan inişine etkisi

Gerçek sıcaklık verisiyle (x ∈ [20, 32]) ölçeklemeden gradyan inişi ne yapar?

In [ ]:
x = np.array([20, 24, 28, 32.]); y = np.array([80, 90, 110, 140.])
print("Ölçeksiz, η=0.001:"); gradyan_inisi(x, y, eta=0.001, iter=2000, yazdir=(1, 100, 2000))
xs = (x - x.mean()) / x.std()
print("Ölçekli,  η=0.1:"); gradyan_inisi(xs, y, eta=0.1, iter=100, yazdir=(1, 10, 100))
print("Ölçekli kapalı çözüm: w =", ((xs)*(y-y.mean())).sum()/(xs**2).sum(), " b =", y.mean())

**Gözlem:** Ölçeksiz veride küçük η bile gerekir ve binlerce iterasyonda hâlâ yavaştır; ölçekli veride 100 iterasyonda çözüme ulaşılır.

## 4. Çoklu regresyon: normal denklem (Örnek 5.3) ve NumPy

In [ ]:
X = np.c_[np.ones(3), [0, 1, 2]]; y3 = np.array([1, 3, 4.])
w_normal = np.linalg.solve(X.T @ X, X.T @ y3)
w_lstsq = np.linalg.lstsq(X, y3, rcond=None)[0]
print("normal denklem:", w_normal.round(3), " lstsq:", w_lstsq.round(3))
print("artıklar:", (y3 - X @ w_normal).round(3), " toplam:", (y3 - X @ w_normal).sum().round(6))

## 5. Dersin verisinde: taban modeli yenmek

4. haftadaki 'saat × gün tipi' profili R² ≈ 0.92 idi. Adım adım özellik ekleyelim.

In [ ]:
profil2 = egitim.groupby(["hafta_sonu", "saat"])["tuketim_kW"].mean()
rapor("Taban: saat x gün tipi", test.tuketim_kW, [profil2[(h, s)] for h, s in zip(test.hafta_sonu, test.saat)])

def ozellik(d, sinCos=True, harmonik=2, klima=True):
    kolon = [d.hafta_sonu.values, d.sicaklik_C.values]
    if klima: kolon.append(np.clip(d.sicaklik_C.values - 24, 0, None))
    if sinCos:
        for k in range(1, harmonik + 1):
            kolon += [np.sin(2*np.pi*k*d.saat.values/24), np.cos(2*np.pi*k*d.saat.values/24)]
    else:
        kolon.append(d.saat.values)
    return np.c_[tuple(kolon)]

for ad, kw in (("saat düz sayı, klima yok", dict(sinCos=False, klima=False)), ("saat düz sayı + klima", dict(sinCos=False)),
               ("sin/cos 1 harmonik + klima", dict(harmonik=1)), ("sin/cos 2 harmonik + klima", dict(harmonik=2)), ("sin/cos 3 harmonik + klima", dict(harmonik=3))):
    m = LinearRegression().fit(ozellik(egitim, **kw), egitim.tuketim_kW)
    rapor(ad, test.tuketim_kW, m.predict(ozellik(test, **kw)))

**Soru:** Hangi adımda tabanı geçtik? Hangi özellik en büyük sıçramayı yaptı? 3 harmonik 2'den iyi mi (aşırı öğrenme başladı mı)?

## 6. Katsayıları yorumlamak

In [ ]:
Xe, Xt = ozellik(egitim), ozellik(test)
isimler = ["hafta_sonu", "T", "klima", "sin1", "cos1", "sin2", "cos2"]
m = LinearRegression().fit(Xe, egitim.tuketim_kW)
print("Ham katsayılar (kW/birim):"); print(pd.Series(m.coef_, isimler).round(2))
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(Xe, egitim.tuketim_kW)
print("\nStandartlaştırılmış katsayılar (kW/σ) — büyüklükler karşılaştırılabilir:")
print(pd.Series(pipe[-1].coef_, isimler).round(2).sort_values(key=abs, ascending=False))

**Yorum:** `klima` katsayısı 24 °C üstündeki her derecenin ek yükünü, `hafta_sonu` katsayısı hafta sonu düşüşünü kW olarak söyler. Standartlaştırılmış katsayılarda sin/cos (günlük profil) en büyüktür: profil, sıcaklıktan daha belirleyici.

## 7. Gerçek–tahmin ve artık analizi

In [ ]:
tahmin = m.predict(Xt); artik = test.tuketim_kW.values - tahmin
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].scatter(test.tuketim_kW, tahmin, s=10); ax[0].plot([0, 260], [0, 260], "k--"); ax[0].set_xlabel("gerçek"); ax[0].set_ylabel("tahmin")
ax[1].hist(artik, bins=25); ax[1].set_xlabel("artık (kW)"); ax[1].set_title(f"ort={artik.mean():.1f}, std={artik.std():.1f}")
ax[2].scatter(test.saat, artik, s=10); ax[2].axhline(0, color="k"); ax[2].set_xlabel("saat"); ax[2].set_ylabel("artık"); ax[2].set_title("Saate göre artık: kalıp var mı?")
plt.tight_layout(); plt.show()

## 8. Ridge ve lasso

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import PolynomialFeatures
# Kasıtlı olarak çok özellik üretelim: 2. derece polinom + etkileşimler (7 -> 36 özellik)
poly = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler())
Xe2, Xt2 = poly.fit_transform(Xe), poly.transform(Xt)
print("özellik sayısı:", Xe2.shape[1])
rapor("Sıradan regresyon (36 özellik)", test.tuketim_kW, LinearRegression().fit(Xe2, egitim.tuketim_kW).predict(Xt2))
for lam in (0.1, 1, 10, 100):
    rapor(f"Ridge λ={lam}", test.tuketim_kW, Ridge(alpha=lam).fit(Xe2, egitim.tuketim_kW).predict(Xt2))
rcv = RidgeCV(alphas=np.logspace(-2, 3, 30), cv=TimeSeriesSplit(5)).fit(Xe2, egitim.tuketim_kW)
rapor(f"RidgeCV (λ={rcv.alpha_:.2f})", test.tuketim_kW, rcv.predict(Xt2))
las = Lasso(alpha=0.5, max_iter=20000).fit(Xe2, egitim.tuketim_kW)
rapor("Lasso α=0.5", test.tuketim_kW, las.predict(Xt2)); print("Lasso sıfırladığı katsayı sayısı:", (las.coef_ == 0).sum(), "/", len(las.coef_))

## 9. Alıştırmalar

**Alıştırma 1.** x = [1, 2, 3], y = [2, 4, 5] için w, b ve R²'yi önce elle (metin hücresinde) sonra kodla bulun.

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `gradyan_inisi` fonksiyonunu çoklu regresyona genelleyin: X (N×d) ve w vektörü; gradyan ∇J = (2/N) Xᵀ(Xw − y). Ders verisinin ölçeklenmiş 7 özelliğinde çalıştırıp `LinearRegression` katsayılarıyla karşılaştırın.

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Etkileşim terimi ekleyin: `klima × gunduz` (gunduz = 8 ≤ saat ≤ 20). Test RMSE değişti mi? Katsayının işareti ne anlatıyor?

In [ ]:
# Alıştırma 3

**Alıştırma 4.** Ridge için λ'yı 0.001'den 1000'e tararken eğitim ve test RMSE'sini çizin (log-x). U biçimini görüyor musunuz? 4. haftadaki karmaşıklık eğrisiyle ilişkisi nedir?

In [ ]:
# Alıştırma 4

**Alıştırma 5 (Ödev 3 başlangıcı).** `pv_uretim.csv`'yi yükleyin, ilk 45 günü eğitim yapın; (a) yalnız ışınım, (b) ışınım + sıcaklık, (c) + rüzgâr + ışınım×sıcaklık modellerinin test RMSE/R² tablosunu üretin ve ışınım katsayısını yorumlayın.

In [ ]:
# Alıştırma 5